# 00 — Exploratory Data Analysis

This notebook tells the first data story for the Fashion Intelligence system. It reads generated evidence; reusable calculations and plots live in `src/fashion/`.

## Scope and rebuild contract

- Modelling evidence uses the shared `train` partition only.
- Validation is used only for normalized development balance and coverage.
- Holdout and quarantine target outcomes stay closed.
- Official prediction images never enter labelled splits, fitted statistics, or modelling EDA.

Rebuild with `./.venv/bin/python scripts/prepare_data.py` followed by `./.venv/bin/python scripts/generate_eda.py`.

In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

from fashion.eda.notebook import figure_html

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

evidence_dir = PROJECT_ROOT / "results/evidence/eda"
figure_dir = PROJECT_ROOT / "results/figures/eda"
summary_path = evidence_dir / "summary.json"
if not summary_path.exists():
    raise FileNotFoundError("Run scripts/prepare_data.py and scripts/generate_eda.py first.")
summary = json.loads(summary_path.read_text(encoding="utf-8"))
target_table = pd.read_csv(evidence_dir / "target_summary.csv")
validation_table = pd.read_csv(evidence_dir / "validation_summary.csv")

## 1. Reconcile raw rows before trusting the split

The raw CSV has 38,617 rows, but five IDs have no valid image, leaving 38,612 image-backed rows. Twenty-one spilled product names are repaired without changing the raw file. Missing image ID 12347 removes the only raw `Suits` example, so the active `articleType` taxonomy has 124 classes rather than the stale 125-class headline. Usage has nine valid classes because literal `NA` is a label; only one raw usage cell is blank.

The split also quarantines all 41 rows in 20 exact-SHA groups with contradictory valid target labels, in addition to cross-role exact duplicates. These are structural safeguards and do not expose holdout or quarantine target distributions.

In [ ]:
reconciliation = summary["data_reconciliation"]
display(pd.Series(reconciliation["raw_to_usable"], name="value").to_frame())
display(
    pd.DataFrame(
        [
            {
                "target": target,
                "stale headline": details["stale_headline_classes"],
                "raw classes": details["raw_valid_classes"],
                "active image-backed classes": details["active_image_backed_classes"],
                "removed by missing images": details["removed_by_missing_images"],
            }
            for target, details in reconciliation["target_taxonomy"].items()
        ]
    )
)

inventory = summary["structural_inventory"]
display(
    pd.DataFrame(
        {
            "partition": inventory["partition_counts"].keys(),
            "rows": inventory["partition_counts"].values(),
        }
    )
)
display(pd.Series(summary["duplicate_and_leakage_control"], name="value").to_frame())

## 2. Article type is the hard long-tail target

The head classes contain thousands of images, while many tail classes have fewer than ten. A single accuracy number will hide these failures. Later comparisons should include macro-F1 and per-class recall, and should state how singleton classes are handled.

In [ ]:
display(
    figure_html(
        figure_dir / "article_type_long_tail.png",
        "Training article type counts and complete class-rank long tail",
    )
)

## 3. The other targets still have imbalance and missing labels

Missing target values are masked, not invented. An image can remain usable for one task while being excluded from the loss and metrics of another. Usage has the sharpest imbalance among the compact targets.

In [ ]:
display(
    figure_html(
        figure_dir / "target_distributions.png",
        "Training distributions for gender, season, and usage, including missing labels",
    )
)
display(target_table)

## 4. Image preprocessing must be consistent

Most images share the same native shape, but rare shapes and grayscale files exist. The shared transform applies EXIF correction, converts to RGB, preserves aspect ratio, adds white letterbox padding, and then uses RGB statistics fitted on training images only.

In [ ]:
display(
    figure_html(
        figure_dir / "image_profile.png",
        "Training image resolutions, file sizes, colour modes, and aspect ratios",
    )
)
display(pd.Series(summary["modelling_evidence"]["normalization"], name="value").to_frame())

## 5. Validation is a development check, not a second training pool

Train and validation counts are unequal, so the comparison uses percentages within each partition. Class names are selected from training only. Coverage below 100% means some training classes have no validation example; this is a limitation, not permission to inspect holdout labels.

In [ ]:
display(
    figure_html(
        figure_dir / "development_balance.png",
        "Train and validation target balance and class coverage using local percentages",
    )
)
display(validation_table)

## Modelling consequences

1. Keep conflicting-label exact-image groups and cross-role twins quarantined.
2. Resolve and test the pending product-group leakage policy before model training.
3. Use `data/processed/splits.csv` for every task and retrieval gallery after that gate is closed.
4. Report macro metrics and tail-class errors, not accuracy alone.
5. Apply target masks before loss and metric calculation.
6. Reuse the same image transform and train-only normalization in training, evaluation, prediction, and the app.
7. Keep the holdout closed until model choices are locked.

The optional perceptual audit can find near-duplicate candidates, but hash collisions are not automatic proof that two fashion images are the same item.